# 05 - Monitoring checks: the first monitored day

The daily monitoring run for `[D, D+1)` fires at the end of day D. It labels **day D−1**, so every hour has had at least 24 h for its crash reports, then measures the forecasts of that day. This notebook reads what the run for 2 January 2023 wrote about **1 January**:

1. performance and fairness (needs the labels);
2. stability (PSI) and feature attribution (SHAP + nDCG), which need no labels;
3. the actions the thresholds triggered.

It ends by checking that every table the pipeline writes has been produced and consumed at least once.

In [1]:
import os

import pandas as pd
import pyarrow.dataset

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")


def table(name):
    return pd.read_parquet(f"datamart/gold/monitoring/{name}")


performance = table("model_performance")
stability = table("stability")
attribution = table("attribution")
actions = table("actions")
labels = pd.read_parquet("datamart/gold/label_store", filters=[("date", "==", "2023-01-01")])
print("labels for 1 January:", len(labels), "rows,", labels["crash_count"].sum(), "crashes,",
      "snapshot", labels["label_snapshot_date"].unique())

labels for 1 January: 1848 rows, 240 crashes, snapshot [datetime.date(2023, 1, 3)]


## 1 - Performance and fairness

One row per metric, model and segment. `citywide` rows come from `monitor_performance`, the group rows from `monitor_fairness`. One day is noisy: the rolling 28-day P0 that drives a retrain needs 28 such days.

In [2]:
citywide = performance[performance["segment"] == "citywide"]
citywide.pivot_table(index="metric", columns="role", values="value").round(3)

role,production,shadow
metric,,
MAE,0.216,0.220
P0,0.296,0.346
RMSE,0.418,0.373
bias,1.080,1.135
poisson_deviance,1.088,0.524


In [3]:
groups = performance[(performance["segment"] != "citywide") & (performance["metric"] == "P0")]
groups.pivot_table(index="segment", columns="role", values="value").round(3)

role,production,shadow
segment,,
crash_rate_high,0.444,0.556
crash_rate_low,0.000,0.000
crash_rate_mid,0.000,0.000
high_taxi_coverage,0.767,0.867
other_areas,0.217,0.286


## 2 - Stability and attribution

**PSI** compares the day's forecasts with the training forecasts (< 0.1 stable, > 0.25 shifted; L7 p. 21). **Attribution** ranks the features by mean |SHAP| on the day, and **nDCG** says how well that ranking keeps the training one (< 0.9 is a violation; SageMaker Clarify).

In [4]:
print(stability[["model_version", "variable", "index", "value", "status"]].round(3).to_string(index=False))

             model_version   variable index  value status
B0_2021_12_20261014T050309 prediction   PSI  0.012     ok
M1_2021_12_20261014T050309 prediction   PSI  0.530  alert


In [5]:
ndcg = attribution[attribution["feature"] == "__ndcg__"][["model_version", "value", "status"]]
print(ndcg.round(3).to_string(index=False))
top = attribution[(attribution["feature"] != "__ndcg__") & (attribution["rank"] <= 5)]
top.pivot_table(index="rank", columns="model_version", values="feature", aggfunc="first")

             model_version  value status
B0_2021_12_20261014T050309  1.000     ok
M1_2021_12_20261014T050309  0.961     ok


model_version,B0_2021_12_20261014T050309,M1_2021_12_20261014T050309
rank,,
1,crash_area_how_mean_4w,crash_area_notified_24h
2,NaN,crash_city_how_mean_4w
3,NaN,wx_visibility_m
4,NaN,crash_city_notified_1h
5,NaN,crash_area_how_mean_4w


## 3 - Actions

`decide_actions` applies one rule of each kind and writes a row for each one that fires.

In [6]:
print(len(actions), "action(s)")
actions

1 action(s)


,at,action,model_version,reason,evidence,date,run_id
0,2026-10-14 08:34:31,alert,M1_2021_12_20261014T050309,forecast distribution shifted (PSI),PSI=0.530,2023-01-01,d_20230102


## 4 - Every table produced and consumed once

Each table, store and the registry has been written by its producer and read by a consumer.

In [7]:
items = {
    "label_store": "datamart/gold/label_store",
    "feature_store": "datamart/gold/feature_store",
    "model_predictions": "datamart/gold/model_predictions",
    "data_quality": "datamart/gold/monitoring/data_quality",
    "model_performance": "datamart/gold/monitoring/model_performance",
    "stability": "datamart/gold/monitoring/stability",
    "attribution": "datamart/gold/monitoring/attribution",
    "ops_run_log": "datamart/gold/monitoring/ops_run_log",
    "actions": "datamart/gold/monitoring/actions",
}
# count rows from the Parquet metadata, without loading the big stores into memory
rows = {name: pyarrow.dataset.dataset(path, partitioning="hive").count_rows() for name, path in items.items()}
rows["registry.json"] = len(pd.read_json("model_bank/registry.json"))
pd.Series(rows, name="rows")

label_store          6474622
feature_store        3268881
model_predictions       7546
data_quality             196
model_performance         30
stability                  2
attribution               22
ops_run_log              392
actions                    1
registry.json              2
Name: rows, dtype: int64

**Findings:**
- **Labels for 1 January:** 77 × 24 = 1,848 rows and 240 crashes. The snapshot is 3 January: the run for 2 January counted every report until the end of that day.
- **Performance, one day:** B0's P0 is 0.296 (0.291 on test), M1's 0.346; MAE about 0.22 for both. Bias above 1 (1.08, 1.14): New Year's Day had fewer crashes than the forecasts. One day is noisy; the retrain rule waits for 28 (`decide_actions`: 1 of 28 days).
- **Fairness:** the P0 gap between the high-taxi-coverage areas and the rest (B0: 0.767 vs 0.217) is about the validation gap, so no bias-drift alert. The low and mid crash-rate terciles never reach the top 10 on a single day (P0 0).
- **Stability:** B0's PSI is 0.012 (stable). **M1's is 0.530, an alert:** one winter holiday's forecasts against the whole training year. This is seasonality, recurring drift in L7's terms, not a broken model. A rolling window or the same season would be a better reference before acting on it.
- **Attribution:** B0's nDCG is 1.000 (one feature); M1's is 0.961, above Clarify's 0.9. Its top features are the area's crashes notified in the last 24 h, the citywide 4-week mean, and visibility.
- **Actions:** one row, the PSI alert on the shadow model.
- **Produced and consumed:** every table has rows: the stores, the forecasts, all six monitoring tables and the registry. (The label store's count includes every snapshot.)